# SpecMine v1.0 sample — worked examples

A short tour of the 500-repository sample: the tool census, spec anatomy, the
spec→code PR layer, and the traceability index. Everything runs on the Parquet
files in `data/` and `specs.jsonl.gz`.

```bash
pip install pandas pyarrow matplotlib
```

In [ ]:
import gzip, json
import pandas as pd

spec_files = pd.read_parquet('data/spec_files.parquet')
features = pd.read_parquet('data/spec_content_features.parquet')
prs = pd.read_parquet('data/pull_requests.parquet')
pr_files = pd.read_parquet('data/pr_files.parquet')
links = pd.read_parquet('data/spec_links.parquet')
kiro = pd.read_parquet('data/kiro_files.parquet')
print(f'{len(spec_files):,} specs across {spec_files.repo_name.nunique():,} repos')

## 1. Which tools produced the specs?

`spec_tool` is the path-fingerprint attribution. Named tools and interpretable
path buckets sit in the same column.

In [ ]:
spec_files.groupby('spec_tool').agg(
    files=('file_url_sha16', 'size'),
    repos=('repo_name', 'nunique')).sort_values('files', ascending=False)

## 2. Adoption over time

Each repository's earliest spec commit dates when it adopted SDD.

In [ ]:
first = (spec_files[spec_files.file_first_commit_at != '']
         .groupby('repo_name').file_first_commit_at.min())
first = pd.to_datetime(first, errors='coerce').dropna()
first.dt.to_period('Q').value_counts().sort_index()

## 3. How big are specs, and what do they contain?

`spec_content_features` has 39 columns per spec (size, structure, requirement
syntax, section markers, quality signals).

In [ ]:
print('median spec length (lines):', int(features.n_lines.median()))
marker_cols = [c for c in features.columns if c.startswith('has_')]
(features[marker_cols].mean().sort_values(ascending=False) * 100).round(1)

## 4. The flagship question: which specs change in the same PR as code?

`touches_code = 1` marks a PR that edits a spec and also changes source files
outside the spec directory. This is the co-change signal the PR layer is built on.

In [ ]:
cochange = prs[prs.touches_code == 1]
print(f'{len(cochange):,} of {len(prs):,} spec-touching PRs also change code '
      f'({100 * len(cochange) / len(prs):.1f}%)')

key = ['tool', 'repo_name', 'pr_number']
changed = pr_files.merge(cochange[key], on=key)
print('spec files changed:', int(changed.is_spec.sum()),
      '| code files changed:', int(changed.is_code.sum()))
cochange.assign(size=cochange.additions + cochange.deletions)[
    ['tool', 'repo_name', 'pr_number', 'n_spec_files', 'n_code_files', 'size']
].sort_values('size', ascending=False).head()

## 5. The traceability index: a second channel from spec to code

`spec_links` records typed references a spec makes to code files, sibling docs,
PRs, issues, and branches, independent of whether they co-change.

In [ ]:
display(links.rel.value_counts())
display(links.provenance.value_counts())

## 6. Reading raw spec text

`specs.jsonl.gz` holds each spec's Markdown, joined on `file_url_sha16`.

In [ ]:
with gzip.open('specs.jsonl.gz', 'rt', encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        if rec['spec_tool'] == 'speckit' and len(rec['content']) > 800:
            print(rec['repo_name'], '::', rec['file_path'])
            print(rec['content'][:800])
            break